In [26]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


StatementMeta(, 0b27dfb3-571c-4ea8-aa0b-f4163e4ccab9, 28, Finished, Available, Finished, False)

In [ ]:
from pyspark.sql import Window
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
from datetime import datetime

# ── Source tables ─────────────────────────────────────────────
SILVER_DAILY  = "silver_daily"
SILVER_5MIN   = "silver_5min"
SILVER_1MIN   = "silver_1min"

# ── Output tables ─────────────────────────────────────────────
GOLD_SECTOR_DAILY     = "gold_sector_daily"
GOLD_TICKER_SIGNALS   = "gold_ticker_signals"
GOLD_RANKINGS_DAILY   = "gold_rankings_daily"
GOLD_CORRELATION      = "gold_correlation_daily"
GOLD_INTRADAY_SUMMARY = "gold_intraday_summary"

# ── Correlation lookback (trading days) ───────────────────────
CORR_WINDOW = 30

# ── Volatility spike threshold (multiplier over rolling avg) ──
# A ticker is flagged if today's volatility is more than 2x
# its own 20-day average volatility — i.e. unusually turbulent.
VOL_SPIKE_MULTIPLIER = 2.0

print("Config defined")

StatementMeta(, 0b27dfb3-571c-4ea8-aa0b-f4163e4ccab9, 29, Finished, Available, Finished, False)

Config defined


In [28]:
# ── STEP 1: READ SILVER ───────────────────────────────────────
# Cache silver_daily — it is used by four of the five gold tables.
# 5min is only needed for the intraday summary.

silver_daily = spark.read.format("delta").table(SILVER_DAILY).cache()
silver_5min  = spark.read.format("delta").table(SILVER_5MIN)

print(f"silver_daily rows : {silver_daily.count():,}")
print(f"silver_5min  rows : {silver_5min.count():,}")

StatementMeta(, 0b27dfb3-571c-4ea8-aa0b-f4163e4ccab9, 30, Finished, Available, Finished, False)

silver_daily rows : 3,416
silver_5min  rows : 56,244


In [31]:
def build_sector_daily(df):
    return (
        df.groupBy("sector", "date", "year", "quarter", "month",
                   "week_of_year", "day_of_week", "is_month_end", "is_quarter_end")
          .agg(
              F.countDistinct("ticker").alias("ticker_count"),

              # Volume-weighted average prices
              F.round(F.sum(F.col("open")  * F.col("volume")) / F.sum("volume"), 4).alias("avg_open"),
              F.round(F.sum(F.col("close") * F.col("volume")) / F.sum("volume"), 4).alias("avg_close"),
              F.round(F.max("high"), 4).alias("sector_high"),
              F.round(F.min("low"),  4).alias("sector_low"),

              F.sum("volume").alias("total_volume"),
              F.sum("transaction_count").alias("total_transactions"),

              # Breadth
              F.count(F.when(F.col("daily_return") > 0,  1)).alias("advancing_tickers"),
              F.count(F.when(F.col("daily_return") < 0,  1)).alias("declining_tickers"),
              F.count(F.when(F.col("daily_return") == 0, 1)).alias("unchanged_tickers"),

              # Averaged indicators
              F.round(F.avg("daily_return"),       6).alias("avg_daily_return"),
              F.round(F.avg("rsi_14"),             2).alias("avg_rsi_14"),
              F.round(F.avg("macd"),               4).alias("avg_macd"),
              F.round(F.avg("rolling_volatility"), 6).alias("avg_volatility"),
              F.round(F.avg("sma_20"),             4).alias("avg_sma_20"),
              F.round(F.avg("sma_50"),             4).alias("avg_sma_50"),

              # Return spread
              F.round(F.max("daily_return") - F.min("daily_return"), 6).alias("return_spread"),
              F.round(F.max("daily_return"), 6).alias("best_return"),
              F.round(F.min("daily_return"), 6).alias("worst_return"),
          )
          .withColumn(
              "advance_decline_ratio",
              F.when(
                  F.col("declining_tickers") == 0, F.lit(None)
              ).otherwise(
                  F.round(F.col("advancing_tickers") / F.col("declining_tickers"), 2)
              )
          )
          .orderBy("sector", "date")
    )

print("build_sector_daily() defined")

StatementMeta(, 0b27dfb3-571c-4ea8-aa0b-f4163e4ccab9, 33, Finished, Available, Finished, False)

build_sector_daily() defined


In [32]:
# ── STEP 3: SIGNAL / ALERT FLAGS ─────────────────────────────
#
# Produces one row per ticker per day with boolean flags for
# five technical signals. Downstream dashboards can filter
# on any combination of flags to surface actionable alerts.
#
# All signals are computed from silver_daily columns — no new
# aggregations, just classification logic applied per row.

def build_ticker_signals(df):
    w = Window.partitionBy("ticker").orderBy("date")

    return (
        df
        # ── 1. RSI overbought / oversold ──────────────────────
        # Classic thresholds: >70 = market may have risen too fast,
        # <30 = may have fallen too far. Neither is a hard sell/buy
        # signal alone — they flag conditions worth watching.
        .withColumn("signal_rsi_overbought",
            F.col("rsi_14") > 70
        )
        .withColumn("signal_rsi_oversold",
            F.col("rsi_14") < 30
        )

        # ── 2. MACD crossover ─────────────────────────────────
        # Bullish crossover: MACD was negative yesterday, positive today
        # (short-term momentum just overtook long-term — potential upswing).
        # Bearish crossover: the reverse.
        .withColumn("_prev_macd", F.lag("macd", 1).over(w))
        .withColumn("signal_macd_bullish_cross",
            (F.col("_prev_macd") < 0) & (F.col("macd") >= 0)
        )
        .withColumn("signal_macd_bearish_cross",
            (F.col("_prev_macd") >= 0) & (F.col("macd") < 0)
        )
        .drop("_prev_macd")

        # ── 3. Bollinger Band touch ───────────────────────────
        # Price at or above upper band = unusually high vs recent range.
        # Price at or below lower band = unusually low.
        .withColumn("signal_bb_upper_touch",
            F.col("close") >= F.col("bb_upper")
        )
        .withColumn("signal_bb_lower_touch",
            F.col("close") <= F.col("bb_lower")
        )

        # ── 4. Golden cross / Death cross ─────────────────────
        # Golden cross: SMA20 crosses above SMA50 (bullish long-term signal).
        # Death cross:  SMA20 crosses below SMA50 (bearish long-term signal).
        # These are rare but high-conviction events — often watched by
        # institutional traders.
        .withColumn("_prev_sma20", F.lag("sma_20", 1).over(w))
        .withColumn("_prev_sma50", F.lag("sma_50", 1).over(w))
        .withColumn("signal_golden_cross",
            (F.col("_prev_sma20") <= F.col("_prev_sma50")) &
            (F.col("sma_20")      >  F.col("sma_50"))
        )
        .withColumn("signal_death_cross",
            (F.col("_prev_sma20") >= F.col("_prev_sma50")) &
            (F.col("sma_20")      <  F.col("sma_50"))
        )
        .drop("_prev_sma20", "_prev_sma50")

        # ── 5. Volatility spike ───────────────────────────────
        # Flag when today's rolling_volatility is more than 2x
        # the ticker's own 20-day average volatility.
        # Uses a separate 20-day window on volatility itself (meta-volatility).
        .withColumn("_avg_vol_20d",
            F.avg("rolling_volatility").over(
                Window.partitionBy("ticker").orderBy("date").rowsBetween(-19, 0)
            )
        )
        .withColumn("signal_vol_spike",
            F.col("rolling_volatility") > (VOL_SPIKE_MULTIPLIER * F.col("_avg_vol_20d"))
        )
        .drop("_avg_vol_20d")

        # ── Convenience: any signal active today ─────────────
        .withColumn("any_signal",
            F.col("signal_rsi_overbought")    |
            F.col("signal_rsi_oversold")      |
            F.col("signal_macd_bullish_cross") |
            F.col("signal_macd_bearish_cross") |
            F.col("signal_bb_upper_touch")    |
            F.col("signal_bb_lower_touch")    |
            F.col("signal_golden_cross")      |
            F.col("signal_death_cross")       |
            F.col("signal_vol_spike")
        )

        # ── Select only identity + signal columns ─────────────
        # Keep the underlying indicator values too — dashboards
        # need them to display context alongside the flag.
        .select(
            "ticker", "sector", "sub_industry", "date",
            "close", "daily_return", "volume",
            "rsi_14", "macd", "sma_20", "sma_50",
            "bb_upper", "bb_lower", "rolling_volatility",
            "signal_rsi_overbought", "signal_rsi_oversold",
            "signal_macd_bullish_cross", "signal_macd_bearish_cross",
            "signal_bb_upper_touch", "signal_bb_lower_touch",
            "signal_golden_cross", "signal_death_cross",
            "signal_vol_spike",
            "any_signal",
        )
        .orderBy("date", "ticker")
    )

print("build_ticker_signals() defined")

StatementMeta(, 0b27dfb3-571c-4ea8-aa0b-f4163e4ccab9, 34, Finished, Available, Finished, False)

build_ticker_signals() defined


In [33]:
# ── STEP 4: DAILY RANKINGS ────────────────────────────────────
#
# Ranks every ticker within its sector on each date across
# four dimensions. Rank 1 = best in sector that day.
#
# Useful for dashboards showing "top movers", "most overbought",
# and "most volatile" within each sector.

def build_rankings_daily(df):
    # Window: rank within sector, per date
    w_sector = Window.partitionBy("sector", "date")

    return (
        df
        # ── Return rank (1 = highest return in sector today) ──
        .withColumn("return_rank_in_sector",
            F.rank().over(w_sector.orderBy(F.col("daily_return").desc()))
        )

        # ── RSI rank (1 = highest RSI — most overbought) ──────
        .withColumn("rsi_rank_in_sector",
            F.rank().over(w_sector.orderBy(F.col("rsi_14").desc()))
        )

        # ── Volatility rank (1 = most volatile) ───────────────
        .withColumn("volatility_rank_in_sector",
            F.rank().over(w_sector.orderBy(F.col("rolling_volatility").desc()))
        )

        # ── Volume rank (1 = highest volume — most traded) ────
        .withColumn("volume_rank_in_sector",
            F.rank().over(w_sector.orderBy(F.col("volume").desc()))
        )

        # ── Overall rank: average of the four ranks above ─────
        # Lower composite score = more notable ticker that day.
        .withColumn("composite_rank_score",
            F.round(
                (
                    F.col("return_rank_in_sector")    +
                    F.col("rsi_rank_in_sector")       +
                    F.col("volatility_rank_in_sector") +
                    F.col("volume_rank_in_sector")
                ) / 4.0, 2
            )
        )

        .select(
            "ticker", "sector", "sub_industry", "date",
            "close", "daily_return", "volume", "rsi_14", "rolling_volatility",
            "return_rank_in_sector", "rsi_rank_in_sector",
            "volatility_rank_in_sector", "volume_rank_in_sector",
            "composite_rank_score",
        )
        .orderBy("date", "sector", "composite_rank_score")
    )

print("build_rankings_daily() defined")

StatementMeta(, 0b27dfb3-571c-4ea8-aa0b-f4163e4ccab9, 35, Finished, Available, Finished, False)

build_rankings_daily() defined


In [34]:
# ── STEP 5: ROLLING CORRELATION TABLE ────────────────────────
#
# Computes the 30-day rolling Pearson correlation of daily_return
# between every pair of tickers within the same sector.
#
# Strategy:
#   1. Pivot silver_daily so each ticker becomes a column of returns
#      (one row per date).
#   2. For each date, collect the trailing 30-day window of returns
#      and compute pairwise correlations using a self-join approach.
#
# Note: Full cross-sector correlations are skipped intentionally —
# within-sector correlations are more actionable for portfolio
# diversification decisions and sector analysis.
#
# Output schema: sector | date | ticker_a | ticker_b | correlation

def build_correlation(df):
    # Keep only what we need
    returns = df.select("ticker", "sector", "date", "daily_return")

    # Self-join on same sector and same date to get all ticker pairs
    left  = returns.alias("a")
    right = returns.alias("b")

    pairs = (
        left.join(right,
            (F.col("a.sector") == F.col("b.sector")) &
            (F.col("a.date")   == F.col("b.date"))   &
            # Only keep a < b to avoid duplicate pairs (AAPL-MSFT and MSFT-AAPL)
            (F.col("a.ticker") < F.col("b.ticker"))
        )
        .select(
            F.col("a.sector").alias("sector"),
            F.col("a.date").alias("date"),
            F.col("a.ticker").alias("ticker_a"),
            F.col("b.ticker").alias("ticker_b"),
            F.col("a.daily_return").alias("return_a"),
            F.col("b.daily_return").alias("return_b"),
        )
    )

    # Rolling 30-day correlation window per ticker pair
    w_corr = (
        Window
        .partitionBy("sector", "ticker_a", "ticker_b")
        .orderBy("date")
        .rowsBetween(-(CORR_WINDOW - 1), 0)
    )

    return (
        pairs
        .withColumn("correlation",
            F.round(F.corr("return_a", "return_b").over(w_corr), 4)
        )
        # Drop rows where correlation could not be computed
        # (fewer than 2 non-null pairs in window)
        .filter(F.col("correlation").isNotNull())
        .select("sector", "date", "ticker_a", "ticker_b", "correlation")
        .orderBy("date", "sector", "ticker_a", "ticker_b")
    )

print("build_correlation() defined")

StatementMeta(, 0b27dfb3-571c-4ea8-aa0b-f4163e4ccab9, 36, Finished, Available, Finished, False)

build_correlation() defined


In [35]:
# ── STEP 6: INTRADAY SESSION SUMMARY ─────────────────────────
#
# Aggregates silver_5min into one row per ticker per date per session
# (pre / regular / after).
#
# Useful for dashboards showing how a stock behaved in pre-market
# vs regular hours — e.g. a big pre-market gap that faded during
# regular hours is a very different story from a steady regular-hours rally.

def build_intraday_summary(df):
    return (
        df.groupBy("ticker", "sector", "sub_industry", "date", "session")
          .agg(
              # OHLCV within the session
              F.first("open",  ignorenulls=True).alias("session_open"),
              F.max("high")                     .alias("session_high"),
              F.min("low")                      .alias("session_low"),
              F.last("close",  ignorenulls=True).alias("session_close"),
              F.sum("volume")                   .alias("session_volume"),
              F.sum("transaction_count")         .alias("session_transactions"),

              # Average VWAP across the session's bars
              F.round(F.avg("vwap"), 4)          .alias("avg_vwap"),

              # Bar count (how many 5-min bars were in this session)
              F.count("*")                       .alias("bar_count"),

              # Average RSI and volatility within session
              F.round(F.avg("rsi_14"),            2).alias("avg_rsi_14"),
              F.round(F.avg("rolling_volatility"), 6).alias("avg_volatility"),
          )
          .withColumn(
              # Session return: how much price moved from session open to close
              "session_return",
              F.round(
                  (F.col("session_close") - F.col("session_open")) / F.col("session_open"), 6
              )
          )
          .orderBy("date", "ticker",
              # Order sessions chronologically
              F.when(F.col("session") == "pre",     1)
               .when(F.col("session") == "regular", 2)
               .when(F.col("session") == "after",   3)
               .otherwise(4)
          )
    )

print("build_intraday_summary() defined")

StatementMeta(, 0b27dfb3-571c-4ea8-aa0b-f4163e4ccab9, 37, Finished, Available, Finished, False)

build_intraday_summary() defined


In [36]:
# ── STEP 7: WRITE GOLD ────────────────────────────────────────
# Same overwrite strategy as silver — gold is always fully
# recomputed from silver, no incremental merges.

def write_gold(df, table_name):
    """Overwrite a gold table and print row count."""
    (
        df.write
          .format("delta")
          .mode("overwrite")
          .option("overwriteSchema", "true")
          .saveAsTable(table_name)
    )
    count = spark.read.format("delta").table(table_name).count()
    print(f"  ✓ {table_name}: {count:,} rows written")

print("write_gold() defined")

StatementMeta(, 0b27dfb3-571c-4ea8-aa0b-f4163e4ccab9, 38, Finished, Available, Finished, False)

write_gold() defined


In [38]:
# ── STEP 8: RUN PIPELINE ──────────────────────────────────────

def run_gold_pipeline():
    start = datetime.utcnow()
    print("=" * 60)
    print("  Gold Layer Pipeline")
    print(f"  Started: {start.strftime('%Y-%m-%d %H:%M:%S')} UTC")
    print("=" * 60)

    # ── Daily-sourced tables ──────────────────────────────────
    print("\n[1/5] Sector daily rollups...")
    write_gold(build_sector_daily(silver_daily),  GOLD_SECTOR_DAILY)

    print("\n[2/5] Ticker signal flags...")
    write_gold(build_ticker_signals(silver_daily), GOLD_TICKER_SIGNALS)

    print("\n[3/5] Daily rankings...")
    write_gold(build_rankings_daily(silver_daily), GOLD_RANKINGS_DAILY)

    print("\n[4/5] Rolling correlations...")
    write_gold(build_correlation(silver_daily),    GOLD_CORRELATION)

    # ── Intraday-sourced table ────────────────────────────────
    print("\n[5/5] Intraday session summary...")
    write_gold(build_intraday_summary(silver_5min), GOLD_INTRADAY_SUMMARY)

    elapsed = (datetime.utcnow() - start).total_seconds() / 60
    print(f"\n✅ Gold pipeline complete in {elapsed:.1f} minutes")


run_gold_pipeline()

StatementMeta(, 0b27dfb3-571c-4ea8-aa0b-f4163e4ccab9, 40, Finished, Available, Finished, False)

  Gold Layer Pipeline
  Started: 2026-06-03 08:58:46 UTC

[1/5] Sector daily rollups...
  ✓ Avi.dbo.gold_sector_daily: 244 rows written

[2/5] Ticker signal flags...
  ✓ Avi.dbo.gold_ticker_signals: 3,416 rows written

[3/5] Daily rankings...
  ✓ Avi.dbo.gold_rankings_daily: 3,416 rows written

[4/5] Rolling correlations...
  ✓ Avi.dbo.gold_correlation_daily: 21,960 rows written

[5/5] Intraday session summary...
  ✓ Avi.dbo.gold_intraday_summary: 1,596 rows written

✅ Gold pipeline complete in 0.6 minutes


In [39]:
# ── SANITY CHECKS ─────────────────────────────────────────────

# 1. Sector daily — check row counts and date coverage per sector
print("── gold_sector_daily ──")
spark.read.format("delta").table(GOLD_SECTOR_DAILY) \
    .groupBy("sector") \
    .agg(
        F.count("*").alias("rows"),
        F.min("date").alias("earliest"),
        F.max("date").alias("latest"),
    ) \
    .orderBy("sector").show()

# 2. Signals — how many flags fired in total across all signal types?
print("── gold_ticker_signals: signal counts ──")
signals_df = spark.read.format("delta").table(GOLD_TICKER_SIGNALS)
for col in [
    "signal_rsi_overbought", "signal_rsi_oversold",
    "signal_macd_bullish_cross", "signal_macd_bearish_cross",
    "signal_bb_upper_touch", "signal_bb_lower_touch",
    "signal_golden_cross", "signal_death_cross", "signal_vol_spike"
]:
    count = signals_df.filter(F.col(col) == True).count()
    print(f"  {col}: {count:,}")

# 3. Rankings — spot check: top 5 tickers by return on the most recent date
print("\n── gold_rankings_daily: top 5 by return on latest date ──")
latest_date = spark.read.format("delta").table(GOLD_RANKINGS_DAILY) \
    .agg(F.max("date")).collect()[0][0]
spark.read.format("delta").table(GOLD_RANKINGS_DAILY) \
    .filter(F.col("date") == latest_date) \
    .orderBy("return_rank_in_sector") \
    .select("ticker", "sector", "daily_return", "return_rank_in_sector", "composite_rank_score") \
    .show(5)

# 4. Correlation — distribution of correlation values
print("── gold_correlation_daily: correlation distribution ──")
spark.read.format("delta").table(GOLD_CORRELATION) \
    .agg(
        F.count("*").alias("pairs"),
        F.round(F.avg("correlation"),  4).alias("mean_corr"),
        F.round(F.min("correlation"),  4).alias("min_corr"),
        F.round(F.max("correlation"),  4).alias("max_corr"),
    ).show()

# 5. Intraday summary — row count per session
print("── gold_intraday_summary: rows per session ──")
spark.read.format("delta").table(GOLD_INTRADAY_SUMMARY) \
    .groupBy("session") \
    .count() \
    .orderBy("session").show()

StatementMeta(, 0b27dfb3-571c-4ea8-aa0b-f4163e4ccab9, 41, Finished, Available, Finished, False)

── gold_sector_daily ──
+-----------+----+----------+----------+
|     sector|rows|  earliest|    latest|
+-----------+----+----------+----------+
|Health Care| 122|2025-12-05|2026-06-02|
|Industrials| 122|2025-12-05|2026-06-02|
+-----------+----+----------+----------+

── gold_ticker_signals: signal counts ──
  signal_rsi_overbought: 524
  signal_rsi_oversold: 476
  signal_macd_bullish_cross: 9
  signal_macd_bearish_cross: 30
  signal_bb_upper_touch: 187
  signal_bb_lower_touch: 175
  signal_golden_cross: 43
  signal_death_cross: 48
  signal_vol_spike: 32

── gold_rankings_daily: top 5 by return on latest date ──
+------+-----------+------------+---------------------+--------------------+
|ticker|     sector|daily_return|return_rank_in_sector|composite_rank_score|
+------+-----------+------------+---------------------+--------------------+
|  ABBV|Health Care|      0.0116|                    1|                5.75|
|  FLGT|Industrials|     0.03372|                    1|               